In [ ]:

import pandas as pd

#read dataset
df = pd.read_csv(r"data/UCI_Credit_Data.csv")

#delete the id colum since it doesn't provide any useful information for the model
df = df.drop("ID", axis=1)

#check the dataset
print(df.head())
print(df.info())
print(df.describe())

#check for missing values
print(df['default.payment.next.month'].value_counts())
print(df['default.payment.next.month'].value_counts(normalize=True))

#check for correlation with the target variable
print(df.corr()['default.payment.next.month'].sort_values(ascending=False))


#create new features based on the existing ones
df['avg_bill_amt'] = df[['BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6']].mean(axis=1)


#credit utilization ratio: avg_bill_amt / LIMIT_BAL
df['credit_utilization'] = df['avg_bill_amt'] / df['LIMIT_BAL']
df['avg_pay_delay'] = df[['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']].mean(axis=1)


#Train/Test Split
from sklearn.model_selection import train_test_split

X = df.drop('default.payment.next.month', axis=1)
y = df['default.payment.next.month']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


#Feature Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


#Model Training - Train Logistic Regression

# from sklearn.linear_model import LogisticRegression
# model = LogisticRegression(max_iter=1000)
# model.fit(X_train_scaled, y_train)

#weight is balanced
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model.fit(X_train_scaled, y_train)

#Model Evaluation - Evaluate Properly
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

#Import and Train XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight= (len(y_train) - sum(y_train)) / sum(y_train),
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost Results:")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))


import matplotlib.pyplot as plt
import pandas as pd

feature_importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(feature_importance.head(10))

feature_importance.head(10).plot(kind='bar')
plt.title("Top 10 Feature Importance - XGBoost")
plt.show()


#Create Main Training Runner

from src.preprocessing import load_data, split_data
from src.feature_engineering import add_engineered_features
from src.train_model import train_xgboost, evaluate_model, save_model

def main():

    df = load_data("data/UCI_Credit_Data.csv")
    df = add_engineered_features(df)

    X_train, X_test, y_train, y_test = split_data(df)

    model = train_xgboost(X_train, y_train)

    evaluate_model(model, X_test, y_test)

    save_model(model)

if __name__ == "__main__":
    main()